In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="funnel-analysis-project-503810")

query = """
    SELECT *
    FROM `funnel-analysis-project-503810.funnel_analysis.funnel_first_touch`
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head()

C:\Users\dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(79193, 6)


,user_pseudo_id,t_page_view,t_view_item,t_add_to_cart,t_begin_checkout,t_purchase
0,1451078.8215732025,2020-11-07 03:03:38.823587+00:00,2020-11-07 03:04:24.373270+00:00,NaT,2020-11-09 11:01:31.401692+00:00,2020-11-09 11:08:26.841419+00:00
1,1714104.6929445672,2020-11-15 09:31:16.450023+00:00,NaT,NaT,NaT,NaT
2,2633877.1089384309,2020-11-15 09:35:48.460907+00:00,NaT,NaT,NaT,NaT
3,2836669.4584691910,2020-11-15 13:46:37.157410+00:00,2020-11-15 13:54:29.135417+00:00,NaT,2020-11-15 14:25:42.302852+00:00,NaT
4,3972359.8765083540,2020-11-15 21:01:42.230260+00:00,2020-11-15 21:01:55.407171+00:00,NaT,NaT,NaT


In [3]:
df['cohort_week'] = df['t_page_view'].dt.to_period('W').dt.start_time
#df['cohort_week'].value_counts().sort_index()
df.head()

C:\Users\dell\AppData\Local\Temp\ipykernel_12056\3600647375.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['cohort_week'] = df['t_page_view'].dt.to_period('W').dt.start_time


,user_pseudo_id,t_page_view,t_view_item,t_add_to_cart,t_begin_checkout,t_purchase,cohort_week
0,1451078.8215732025,2020-11-07 03:03:38.823587+00:00,2020-11-07 03:04:24.373270+00:00,NaT,2020-11-09 11:01:31.401692+00:00,2020-11-09 11:08:26.841419+00:00,2020-11-02
1,1714104.6929445672,2020-11-15 09:31:16.450023+00:00,NaT,NaT,NaT,NaT,2020-11-09
2,2633877.1089384309,2020-11-15 09:35:48.460907+00:00,NaT,NaT,NaT,NaT,2020-11-09
3,2836669.4584691910,2020-11-15 13:46:37.157410+00:00,2020-11-15 13:54:29.135417+00:00,NaT,2020-11-15 14:25:42.302852+00:00,NaT,2020-11-09
4,3972359.8765083540,2020-11-15 21:01:42.230260+00:00,2020-11-15 21:01:55.407171+00:00,NaT,NaT,NaT,2020-11-09


In [4]:
df['cohort_week'].value_counts().sort_index()

cohort_week
2020-10-26     2302
2020-11-02    19258
2020-11-09    16214
2020-11-16    18157
2020-11-23    20044
2020-11-30     3206
Name: count, dtype: int64

In [5]:
stages = ['t_page_view','t_view_item','t_add_to_cart','t_begin_checkout','t_purchase']

cohort_counts = df.groupby('cohort_week')[stages].apply(lambda x: x.notna().sum())
cohort_counts

,t_page_view,t_view_item,t_add_to_cart,t_begin_checkout,t_purchase
cohort_week,,,,,
2020-10-26,2302,549,7,114,33
2020-11-02,19258,5142,85,1022,364
2020-11-09,16214,4613,90,1013,373
2020-11-16,18157,5205,515,890,315
2020-11-23,20044,5043,1053,1013,370
2020-11-30,3206,876,310,167,77


In [6]:
overall_conversion = cohort_counts.div(cohort_counts['t_page_view'], axis=0) * 100
overall_conversion.round(1)

,t_page_view,t_view_item,t_add_to_cart,t_begin_checkout,t_purchase
cohort_week,,,,,
2020-10-26,100.0,23.8,0.3,5.0,1.4
2020-11-02,100.0,26.7,0.4,5.3,1.9
2020-11-09,100.0,28.5,0.6,6.2,2.3
2020-11-16,100.0,28.7,2.8,4.9,1.7
2020-11-23,100.0,25.2,5.3,5.1,1.8
2020-11-30,100.0,27.3,9.7,5.2,2.4


#### Sequential funnel

In [7]:
import plotly.express as px

fig = px.imshow(
    overall_conversion,
    labels=dict(x="Funnel Stage", y="Cohort Week", color="Conversion %"),
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto"
)
fig.update_layout(title="Conversion Rate by Cohort Week (%)")
fig.show()

In [8]:
from google.cloud import bigquery

client = bigquery.Client(project="funnel-analysis-project-503810")

query = """
    SELECT *
    FROM `funnel-analysis-project-503810.funnel_analysis.funnel_sequential`
"""

df_seq = client.query(query).to_dataframe()

print(df_seq.shape)
df_seq.head()

C:\Users\dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(79193, 6)


,user_pseudo_id,t_page_view,t_view_item_seq,t_add_to_cart_seq,t_begin_checkout_seq,t_purchase_seq
0,71120941.6331406898,2020-11-09 08:50:13.664512+00:00,2020-11-09 08:50:38.232952+00:00,NaT,NaT,NaT
1,9094524.2128918873,2020-11-14 00:28:20.147445+00:00,2020-11-14 00:35:38.681808+00:00,NaT,NaT,NaT
2,53489490.6356830127,2020-11-19 00:16:07.107529+00:00,NaT,NaT,NaT,NaT
3,8150555.1205669386,2020-11-19 08:59:33.201889+00:00,2020-11-19 09:00:14.638446+00:00,NaT,NaT,NaT
4,3072698.2077162467,2020-11-04 18:47:30.550698+00:00,2020-11-04 18:47:35.204010+00:00,NaT,NaT,NaT


In [9]:
import plotly.graph_objects as go

stage_labels = ['Page View', 'View Item', 'Add to Cart', 'Begin Checkout', 'Purchase']
open_counts = [79181, 21440, 2060, 4219, 1532]
sequential_counts = [79181, 21316, 1850, 694, 344]

fig = go.Figure()
fig.add_trace(go.Funnel(name='Open (first-touch)', y=stage_labels, x=open_counts, textinfo="value+percent initial"))
fig.add_trace(go.Funnel(name='Sequential (strict order)', y=stage_labels, x=sequential_counts, textinfo="value+percent initial"))
fig.update_layout(title='Open vs. Sequential Funnel Comparison')
fig.show()